In [ ]:
# =========================================
# ADIM 3: MODELİ YÜKLE VE TAHMİN YAP (NameError Korumalı)
# =========================================
import os
import numpy as np
import h2o
from sklearn.metrics import precision_score, recall_score, f1_score as f1_func

# 1. H2O motorunu garantile
h2o.init(verbose=False)

# 2. Modeli diskten bul ve 'best_model' adıyla RAM'e yükle (NameError'ı çözen kısım!)
save_dir = "./best_food_model"
model_filename = [f for f in os.listdir(save_dir) if not f.startswith('.')][0]
best_model = h2o.load_model(os.path.join(save_dir, model_filename))
print(f"✅ Model başarıyla hafızaya alındı: {best_model.model_id}")

# 3. Test verisi üzerinde tahminleri çalıştır
print("⚙️ Test seti üzerinde tahminler yapılıyor...")
perf_test = best_model.model_performance(test_h2o)
pred_h2o = best_model.predict(test_h2o)

pred_df  = pred_h2o.as_data_frame()
true_df  = test_h2o["label"].as_data_frame()

# 4. Veri tiplerini güvenli şekilde dönüştür
y_true = true_df["label"].astype(float).astype(int).astype(str).values
y_pred = pred_df["predict"].astype(float).astype(int).astype(str).values

# 5. Metrikleri Hesapla
accuracy = np.mean(y_pred == y_true)
precision = precision_score(y_true, y_pred, average='macro', zero_division=0)
recall = recall_score(y_true, y_pred, average='macro', zero_division=0)
f1_score = f1_func(y_true, y_pred, average='macro', zero_division=0)

# 6. Kusursuz Top-5 Accuracy Hesaplaması
h2o_classes = [str(int(float(c))) for c in test_h2o["label"].levels()[0]]
prob_df = pred_df.drop("predict", axis=1)
probs = prob_df.values
top5_indices = np.argsort(probs, axis=1)[:, -5:]

top5_acc_count = 0
for i in range(len(y_true)):
    top5_classes_for_i = [h2o_classes[idx] for idx in top5_indices[i]]
    if y_true[i] in top5_classes_for_i:
        top5_acc_count += 1

top5_accuracy = top5_acc_count / len(y_true)

# =========================================
# ADIM 4: PERFORMANS KARNESİ VE KAPANIŞ
# =========================================

# Görseldeki tasarıma birebir uygun Performans Karnesi
print("\n" + "=" * 60)
print("🚀 MODELİN DETAYLI PERFORMANS KARNESİ")
print("=" * 60)
print(f"Top-1 Başarı Oranı (Accuracy): %{accuracy * 100:.2f}")
print(f"Top-5 Başarı Oranı (Esnek):    %{top5_accuracy * 100:.2f}")
print("-" * 60)
print(f"Precision (Kesinlik):          %{precision * 100:.2f}")
print(f"Recall (Duyarlılık):           %{recall * 100:.2f}")
print(f"F1-Score (Genel Skor):         %{f1_score * 100:.2f}")
print("=" * 60)

print(f"\nTest Logloss            : {perf_test.logloss():.4f}")
print(f"Test Mean Per Class Err : {perf_test.mean_per_class_error():.4f}")

h2o.cluster().shutdown(prompt=False)
print("\n✅ İşlemler başarıyla tamamlandı ve H2O sunucusu kapatıldı!")


FINAL PERFORMANS — TEST SETİ (model bunu hiç görmedi):


NameError: name 'best_model' is not defined

In [14]:
# =========================================
# ADIM 3: MODELİ YÜKLE VE TAHMİN YAP (NameError Korumalı)
# =========================================
import os
import math
import numpy as np
import tensorflow as tf
import kagglehub
from tqdm import tqdm
import pandas as pd
import h2o
from h2o.automl import H2OAutoML
from sklearn.metrics import precision_score, recall_score, f1_score as f1_func
import os
import numpy as np
import pandas as pd
import h2o
from sklearn.metrics import precision_score, recall_score, f1_score as f1_func
# =========================================
# ADIM 3: TEST VERİSİNİ VE MODELİ YÜKLE, TAHMİN YAP
# =========================================

# 1. H2O motorunu garantile
h2o.init(verbose=False)

# 2. Modeli diskten bul ve 'best_model' adıyla RAM'e yükle
save_dir = "./best_food_model"
model_filename = [f for f in os.listdir(save_dir) if not f.startswith('.')][0]
best_model = h2o.load_model(os.path.join(save_dir, model_filename))
print(f"✅ Model başarıyla hafızaya alındı: {best_model.model_id}")

# 3. YENİ EKLENEN KISIM: Test Verisini Diskten RAM'e Yükle
print("📥 Test verisi 'features' klasöründen yükleniyor...")
X_test = np.load("./features/test_features.npy")
y_test = np.load("./features/test_labels.npy")

# HAYAT KURTARAN DÜZELTME: Sütunlara f0, f1... isimlerini veriyoruz ki model onları tanısın!
feature_cols = [f"f{i}" for i in range(X_test.shape[1])]
test_df = pd.DataFrame(X_test, columns=feature_cols)

test_df["label"] = y_test
test_h2o = h2o.H2OFrame(test_df)
test_h2o["label"] = test_h2o["label"].asfactor() # Sınıflandırma için zorunlu
print(f"✅ Test verisi başarıyla H2O'ya aktarıldı. (Satır, Sütun): {test_h2o.shape}")

# 4. Test verisi üzerinde tahminleri çalıştır
print("\n⚙️ Test seti üzerinde tahminler yapılıyor...")
perf_test = best_model.model_performance(test_h2o)
pred_h2o = best_model.predict(test_h2o)

pred_df  = pred_h2o.as_data_frame()
true_df  = test_h2o["label"].as_data_frame()

# 5. Veri tiplerini güvenli şekilde dönüştür
y_true = true_df["label"].astype(float).astype(int).astype(str).values
y_pred = pred_df["predict"].astype(float).astype(int).astype(str).values

# 6. Metrikleri Hesapla
accuracy = np.mean(y_pred == y_true)
precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
f1_score = f1_func(y_true, y_pred, average='weighted', zero_division=0)

# 7. Kusursuz Top-5 Accuracy Hesaplaması
prob_df = pred_df.drop("predict", axis=1)

# ÇÖZÜM: H2O'nun eklediği 'p' harfini siliyoruz ('p0' -> '0')
# Ardından aynı güvenli float -> int -> str dönüşümünü uyguluyoruz.
h2o_classes = [str(int(float(c.replace('p', '')))) for c in prob_df.columns]

probs = prob_df.values
top5_indices = np.argsort(probs, axis=1)[:, -5:]

top5_acc_count = 0
for i in range(len(y_true)):
    top5_classes_for_i = [h2o_classes[idx] for idx in top5_indices[i]]
    if y_true[i] in top5_classes_for_i:
        top5_acc_count += 1

top5_accuracy = top5_acc_count / len(y_true)

# =========================================
# ADIM 4: PERFORMANS KARNESİ VE KAPANIŞ
# =========================================

# Görseldeki tasarıma birebir uygun Performans Karnesi
print("\n" + "=" * 60)
print("🚀 MODELİN DETAYLI PERFORMANS KARNESİ")
print("=" * 60)
print(f"Top-1 Başarı Oranı (Accuracy): %{accuracy * 100:.2f}")
print(f"Top-5 Başarı Oranı (Esnek):    %{top5_accuracy * 100:.2f}")
print("-" * 60)
print(f"Precision (Kesinlik):          %{precision * 100:.2f}")
print(f"Recall (Duyarlılık):           %{recall * 100:.2f}")
print(f"F1-Score (Genel Skor):         %{f1_score * 100:.2f}")
print("=" * 60)

print(f"\nTest Logloss            : {perf_test.logloss():.4f}")
print(f"Test Mean Per Class Err : {perf_test.mean_per_class_error():.4f}")

h2o.cluster().shutdown(prompt=False)
print("\n✅ İşlemler başarıyla tamamlandı ve H2O sunucusu kapatıldı!")

✅ Model başarıyla hafızaya alındı: GLM_1_AutoML_1_20260324_15106
📥 Test verisi 'features' klasöründen yükleniyor...
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
✅ Test verisi başarıyla H2O'ya aktarıldı. (Satır, Sütun): (15150, 1281)

⚙️ Test seti üzerinde tahminler yapılıyor...
glm prediction progress: |███████████████████████████████████████████████████████| (done) 100%


/Users/semra/VsCodeProjects/5ml_task/food-101_classification/venv_ml/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"
/Users/semra/VsCodeProjects/5ml_task/food-101_classification/venv_ml/lib/python3.12/site-packages/h2o/frame.py:1983: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  warnings.warn("Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using"



🚀 MODELİN DETAYLI PERFORMANS KARNESİ
Top-1 Başarı Oranı (Accuracy): %84.42
Top-5 Başarı Oranı (Esnek):    %95.45
------------------------------------------------------------
Precision (Kesinlik):          %97.31
Recall (Duyarlılık):           %84.42
F1-Score (Genel Skor):         %90.12

Test Logloss            : 0.7059
Test Mean Per Class Err : 0.0243

✅ İşlemler başarıyla tamamlandı ve H2O sunucusu kapatıldı!
